# 00 · Colab Setup

Mount Drive, configure paths, install deps, and download the Kaggle dataset.
**Run this once per Colab session before any other notebook.**

Dataset: [pkdarabi/cardetection](https://www.kaggle.com/datasets/pkdarabi/cardetection/data) — *Traffic Signs Detection* (15 classes, required teacher dataset).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set repo path + make `src` importable

In [ ]:
import os, sys
REPO = '/content/drive/MyDrive/HCMUE_Projects/DeepLearning/traffic-sign-detection-yolo-detr'
os.environ['TSD_REPO_ROOT'] = REPO
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('repo root:', REPO)

## 3. Install dependencies

In [ ]:
import os
REPO = os.environ['TSD_REPO_ROOT']
!pip install -q -r {REPO}/requirements.txt

## 4. Kaggle credentials

`kaggle.json` is stored in the repo folder on Drive (same as `TSD_REPO_ROOT`). We copy it to `/root/.kaggle/` where the Kaggle CLI expects it.

In [ ]:
import os
REPO = os.environ['TSD_REPO_ROOT']
os.makedirs('/root/.kaggle', exist_ok=True)
!cp {REPO}/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('kaggle.json installed')

## 5. Download the dataset

In [ ]:
import os
REPO = os.environ['TSD_REPO_ROOT']
!kaggle datasets download -d pkdarabi/cardetection -p {REPO}/data/raw --unzip
!ls {REPO}/data/raw

## 6. Organise into `data/processed/cardetection/`

The Roboflow export unzips to a single subfolder (e.g. `data/raw/cardetection/` or similar). This cell finds it and symlinks it to `data/processed/cardetection` so all scripts resolve to the same root.

In [ ]:
import os, pathlib
REPO = pathlib.Path(os.environ['TSD_REPO_ROOT'])
raw = REPO / 'data' / 'raw'
processed = REPO / 'data' / 'processed' / 'cardetection'

# Find the subfolder that contains data.yaml (the Roboflow export root).
export_root = None
for candidate in [raw, *raw.iterdir()]:
    if (candidate / 'data.yaml').exists():
        export_root = candidate
        break

if export_root is None:
    raise FileNotFoundError('Could not locate data.yaml under data/raw — check the download.')

processed.parent.mkdir(parents=True, exist_ok=True)
if not processed.exists():
    processed.symlink_to(export_root.resolve())
    print(f'symlinked {processed} -> {export_root}')
else:
    print(f'already exists: {processed}')

# Quick sanity check
for split in ('train', 'valid', 'test'):
    imgs = list((processed / split / 'images').glob('*')) if (processed / split / 'images').exists() else []
    print(f'  {split}: {len(imgs)} images')

## 7. Verify: inspect the dataset

In [ ]:
from src.data.inspect_dataset import inspect
from src.utils.paths import DATA_PROCESSED
inspect(DATA_PROCESSED)